# Autodiff: The Key Idea

**Time: ~25 minutes**

Automatic differentiation is what makes PINNs possible. This notebook builds your intuition for `torch.autograd.grad` — the function you'll use in every PINN you build.

In [ ]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(42)

## Three Ways to Compute Derivatives

| Method | Pros | Cons |
|--------|------|------|
| **Symbolic** (SymPy) | Exact formula | Doesn't work for neural networks |
| **Finite differences** | Simple | Approximate, numerically unstable |
| **Automatic differentiation** | Exact, works for any differentiable program | Requires framework support |

PINNs need **exact** derivatives of a neural network. Autodiff is the only option.

## Warm-up: Differentiating Simple Functions

Let's start with `f(x) = x² + 3x`. We know `f'(x) = 2x + 3`.

In [ ]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

f = x**2 + 3*x  # f(x) = x² + 3x

# torch.autograd.grad computes df/dx
dfdx = torch.autograd.grad(
    outputs=f,           # what to differentiate
    inputs=x,            # with respect to what
    grad_outputs=torch.ones_like(f),  # "seed" — needed for non-scalar outputs
    create_graph=True,   # keep the computation graph for higher-order derivatives
)[0]

print(f"x:       {x.detach()}")
print(f"f(x):    {f.detach()}")
print(f"f'(x):   {dfdx.detach()}")
print(f"Expected: {(2*x + 3).detach()}  (2x + 3)")

### The `grad_outputs` parameter

When the output is not a scalar, PyTorch needs `grad_outputs` to know "which direction" to differentiate. For PINNs, we always pass `torch.ones_like(output)` — this gives us the elementwise derivative, which is what we want.

### The `create_graph=True` parameter

This is **critical** for PINNs. It tells PyTorch to keep the computation graph alive so we can:
1. Compute **higher-order derivatives** (d²u/dx²) by differentiating the derivative
2. **Backpropagate through the derivative** during training

## Higher-Order Derivatives

Many PDEs need second derivatives (diffusion, wave equation). Just differentiate the derivative:

In [ ]:
x = torch.linspace(0, 2*3.14159, 50, requires_grad=True).unsqueeze(1)

f = torch.sin(x)  # f(x) = sin(x)

# First derivative: f'(x) = cos(x)
dfdx = torch.autograd.grad(f, x, torch.ones_like(f), create_graph=True)[0]

# Second derivative: f''(x) = -sin(x)
d2fdx2 = torch.autograd.grad(dfdx, x, torch.ones_like(dfdx), create_graph=True)[0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xn = x.detach().numpy()
axes[0].plot(xn, f.detach().numpy(), 'b-'); axes[0].set_title("f(x) = sin(x)")
axes[1].plot(xn, dfdx.detach().numpy(), 'r-'); axes[1].set_title("f'(x) = cos(x)")
axes[2].plot(xn, d2fdx2.detach().numpy(), 'g-'); axes[2].set_title("f''(x) = -sin(x)")
for ax in axes: ax.grid(True, alpha=0.3); ax.set_xlabel("x")
plt.tight_layout(); plt.show()

## Differentiating a Neural Network

This is where it gets interesting. A neural network is just a composition of differentiable operations, so autograd works on it directly.

In [ ]:
net = nn.Sequential(
    nn.Linear(1, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
)

x = torch.linspace(0, 1, 100).unsqueeze(1).requires_grad_(True)
u = net(x)

# Compute du/dx
du_dx = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]

# Compute d²u/dx²
d2u_dx2 = torch.autograd.grad(du_dx, x, torch.ones_like(du_dx), create_graph=True)[0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xn = x.detach().numpy()
axes[0].plot(xn, u.detach().numpy(), 'b-'); axes[0].set_title("u(x) — network output")
axes[1].plot(xn, du_dx.detach().numpy(), 'r-'); axes[1].set_title("du/dx — first derivative")
axes[2].plot(xn, d2u_dx2.detach().numpy(), 'g-'); axes[2].set_title("d²u/dx² — second derivative")
for ax in axes: ax.grid(True, alpha=0.3); ax.set_xlabel("x")
plt.tight_layout(); plt.show()

print("All three are smooth because we used tanh activation.")
print("ReLU would give piecewise-linear u, piecewise-constant du/dx, and zero d²u/dx².")

## Why Tanh? Activation Functions Matter

PINNs typically need second (or higher) derivatives. The activation function determines whether these exist and are useful:

| Activation | u | du/dx | d²u/dx² | PINN-friendly? |
|-----------|---|-------|---------|----------------|
| ReLU | Piecewise linear | Piecewise constant | Zero everywhere | No |
| Tanh | Smooth | Smooth | Smooth | Yes |
| Sin | Smooth | Smooth | Smooth | Yes |
| GELU | Smooth | Smooth | Smooth | Sometimes |

**Rule: always use `tanh` (or `sin`) for PINNs.** The `pinn.PINN` class in this repo uses `tanh` by default.

In [ ]:
# Demonstration: ReLU destroys second derivatives
relu_net = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(),
    nn.Linear(64, 64), nn.ReLU(),
    nn.Linear(64, 1),
)

x = torch.linspace(0, 1, 200).unsqueeze(1).requires_grad_(True)
u_relu = relu_net(x)
du_relu = torch.autograd.grad(u_relu, x, torch.ones_like(u_relu), create_graph=True)[0]
d2u_relu = torch.autograd.grad(du_relu, x, torch.ones_like(du_relu), create_graph=True)[0]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
xn = x.detach().numpy()
axes[0].plot(xn, u_relu.detach().numpy(), 'b-'); axes[0].set_title("u(x) — ReLU net")
axes[1].plot(xn, du_relu.detach().numpy(), 'r-'); axes[1].set_title("du/dx — piecewise constant")
axes[2].plot(xn, d2u_relu.detach().numpy(), 'g-'); axes[2].set_title("d²u/dx² — zero!")
for ax in axes: ax.grid(True, alpha=0.3); ax.set_xlabel("x")
plt.suptitle("ReLU: second derivative is zero almost everywhere — useless for PDEs", fontweight='bold')
plt.tight_layout(); plt.show()

## Multivariate Derivatives (Partial Derivatives)

PDEs have multiple independent variables. The key: give each variable its own tensor with `requires_grad=True`.

In [ ]:
# Network taking (x, t) as input
net2d = nn.Sequential(
    nn.Linear(2, 32), nn.Tanh(),
    nn.Linear(32, 32), nn.Tanh(),
    nn.Linear(32, 1),
)

# Create SEPARATE tensors for x and t — this is essential
x = torch.rand(100, 1, requires_grad=True)
t = torch.rand(100, 1, requires_grad=True)

# Forward pass: concatenate inputs
u = net2d(torch.cat([x, t], dim=1))

# Partial derivatives — just specify which input variable
du_dx = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
du_dt = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]

# Second partial derivative: d²u/dx²
d2u_dx2 = torch.autograd.grad(du_dx, x, torch.ones_like(du_dx), create_graph=True)[0]

print(f"u shape:       {u.shape}")
print(f"du/dx shape:   {du_dx.shape}")
print(f"du/dt shape:   {du_dt.shape}")
print(f"d²u/dx² shape: {d2u_dx2.shape}")
print(f"\nAll shapes match — each derivative is computed pointwise.")

In [ ]:
# Now we can build a PDE residual!
# Heat equation: du/dt = nu * d²u/dx²
nu = 0.01
residual = du_dt - nu * d2u_dx2
physics_loss = torch.mean(residual**2)

print(f"Heat equation residual (untrained): {physics_loss.item():.6f}")
print("\nTraining would minimize this — driving the network toward the true solution.")

## The Pattern You'll Use in Every PINN

```python
# 1. Create input tensors with requires_grad=True
x = torch.rand(N, 1, requires_grad=True)
t = torch.rand(N, 1, requires_grad=True)

# 2. Forward pass
u = model(torch.cat([x, t], dim=1))

# 3. Compute needed derivatives
u_t = torch.autograd.grad(u, t, torch.ones_like(u), create_graph=True)[0]
u_x = torch.autograd.grad(u, x, torch.ones_like(u), create_graph=True)[0]
u_xx = torch.autograd.grad(u_x, x, torch.ones_like(u_x), create_graph=True)[0]

# 4. Build residual from the PDE
residual = u_t - nu * u_xx  # heat equation
loss = torch.mean(residual**2)
```

This pattern is the same for every PDE — only the residual formula changes.

## Exercise: Compute the Burgers' Equation Residual

Burgers' equation: `u_t + u * u_x = nu * u_xx`

Rearranged as residual: `u_t + u * u_x - nu * u_xx = 0`

Try writing the residual using the derivatives we already computed.

In [ ]:
# Exercise: fill in the Burgers residual
# Hint: you already have u, du_dt, du_dx (= du_dx from earlier), d2u_dx2

nu = 0.01
burgers_residual = du_dt + u * du_dx - nu * d2u_dx2  # Burgers' equation
burgers_loss = torch.mean(burgers_residual**2)

print(f"Burgers residual (untrained): {burgers_loss.item():.6f}")
print("Same pattern — different equation, same autograd machinery.")

## Key Takeaways

1. **`torch.autograd.grad`** computes exact derivatives of any differentiable computation
2. **`create_graph=True`** is essential — it enables higher-order derivatives and backprop through the derivative
3. **`grad_outputs=torch.ones_like(output)`** gives elementwise derivatives for non-scalar outputs
4. **Separate input tensors** (each with `requires_grad=True`) give you partial derivatives
5. **Use `tanh` activation** — ReLU kills second derivatives
6. The **pattern is universal**: compute derivatives → build residual → minimize

## What's Next

**Notebook 03** puts this all together: we'll build a complete PINN from scratch, train it on `u' = -u`, and watch it converge to the exact solution `u = exp(-t)`.